In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn_prg import average_precision_recall_gain

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'data').is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

MATRIX_DIR = PROJECT_DIR / 'data'
if not (MATRIX_DIR / 'fastani-upper-triangle.parquet').is_file():
    MATRIX_DIR = PROJECT_DIR.parents[1] / PROJECT_DIR.name / 'data'

ANI_PATH = MATRIX_DIR / 'fastani-upper-triangle.parquet'
AF_PATH = MATRIX_DIR / 'fastani-AF-upper-triangle.parquet'
AAI_PATH = MATRIX_DIR / 'AAI-upper-triangle.parquet'
METADATA_PATH = PROJECT_DIR / 'data' / 'genome_tax_metadata.parquet'
MARKER_PAIRS_PATH = PROJECT_DIR / 'tmp_data' / 'marker-pairwise-results.parquet'
RESULTS_PATH = PROJECT_DIR / 'figs' / 'auprg_by_family_sklearn_prg.tsv'

#ANI_MIN = 75.0
#AF_MIN = 0.20
MARKER_IDENTITY_MIN = 50.0
ITS_ALIGNMENT_LENGTH_MIN = 50
FIVE_EIGHT_S_ALIGNMENT_LENGTH_MIN = 150
MIN_SPECIES_REPRESENTATIVES = 10
METRICS = ['ANI', 'AAI', 'ITS1', 'ITS2', '5.8S']


def sql_path(path):
    return str(path).replace("'", "''")


required_paths = (
    ANI_PATH, AF_PATH, AAI_PATH, METADATA_PATH, MARKER_PAIRS_PATH
)
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('Missing input file(s):\n' + '\n'.join(map(str, missing_paths)))


In [2]:
def load_filtered_pairs():
    con = duckdb.connect()
    try:
        con.execute('SET threads=4')
        con.execute('SET max_expression_depth=100000')
        return con.execute(
            f"""
            WITH ani_long AS (
                SELECT assembly_accession AS genome1,
                       compared_accession AS genome2,
                       value::DOUBLE AS ANI
                FROM (
                    UNPIVOT read_parquet('{sql_path(ANI_PATH)}')
                    ON COLUMNS(* EXCLUDE (assembly_accession))
                    INTO NAME compared_accession VALUE value
                )
                WHERE assembly_accession <> compared_accession
                  AND value IS NOT NULL AND isfinite(value::DOUBLE)
            ),
            af_long AS (
                SELECT assembly_accession AS genome1,
                       compared_accession AS genome2,
                       value::DOUBLE AS AF
                FROM (
                    UNPIVOT read_parquet('{sql_path(AF_PATH)}')
                    ON COLUMNS(* EXCLUDE (assembly_accession))
                    INTO NAME compared_accession VALUE value
                )
                WHERE assembly_accession <> compared_accession
                  AND value IS NOT NULL AND isfinite(value::DOUBLE)
            ),
            aai_long AS (
                SELECT assembly_accession AS genome1,
                       compared_accession AS genome2,
                       value::DOUBLE AS AAI
                FROM (
                    UNPIVOT read_parquet('{sql_path(AAI_PATH)}')
                    ON COLUMNS(* EXCLUDE (assembly_accession))
                    INTO NAME compared_accession VALUE value
                )
                WHERE assembly_accession <> compared_accession
                  AND value IS NOT NULL AND isfinite(value::DOUBLE)
            ),
            retained_metadata AS (
                SELECT ncbi_genome_accession AS genome,
                       phylum_2026_09_01 AS phylum,
                       family_2026_09_01 AS family,
                       species_2026_09_01 AS species
                FROM read_parquet('{sql_path(METADATA_PATH)}')
                WHERE qc = 'pass'
                  AND NOT coalesce(drop_dup, false)
                  AND phylum_2026_09_01 IS NOT NULL
                  AND family_2026_09_01 IS NOT NULL
                  AND species_2026_09_01 IS NOT NULL
                  AND trim(phylum_2026_09_01) <> ''
                  AND trim(family_2026_09_01) <> ''
                  AND trim(species_2026_09_01) <> ''
            ),
            eligible_species AS (
                SELECT species
                FROM retained_metadata
                GROUP BY species
                HAVING count(*) >= {MIN_SPECIES_REPRESENTATIVES}
            ),
            retained AS (
                SELECT m.*
                FROM retained_metadata m
                JOIN eligible_species e USING (species)
            ),
            base_pairs AS (
                SELECT a.genome1, a.genome2, a.ANI, x.AAI,
                       m1.phylum, m1.family,
                       (m1.species = m2.species)::INTEGER AS same_species
                FROM ani_long a
                JOIN af_long f USING (genome1, genome2)
                LEFT JOIN aai_long x USING (genome1, genome2)
                JOIN retained m1 ON a.genome1 = m1.genome
                JOIN retained m2 ON a.genome2 = m2.genome
                WHERE m1.phylum = m2.phylum
                  AND m1.family = m2.family
            ),
            marker_best AS (
                SELECT genome1, genome2, region, marker_identity
                FROM read_parquet('{sql_path(MARKER_PAIRS_PATH)}')
                WHERE marker_identity BETWEEN {MARKER_IDENTITY_MIN} AND 100
                  AND (
                        (region IN ('ITS1', 'ITS2')
                         AND alignment_length >= {ITS_ALIGNMENT_LENGTH_MIN})
                     OR (region = '5.8S'
                         AND alignment_length >= {FIVE_EIGHT_S_ALIGNMENT_LENGTH_MIN})
                  )
            ),
            its_wide AS (
                SELECT genome1, genome2,
                       max(marker_identity) FILTER (WHERE region = 'ITS1') AS ITS1,
                       max(marker_identity) FILTER (WHERE region = 'ITS2') AS ITS2,
                       max(marker_identity) FILTER (WHERE region = '5.8S') AS "5.8S"
                FROM marker_best
                GROUP BY genome1, genome2
            )
            SELECT p.phylum, p.family, p.same_species, p.ANI, p.AAI,
                   i.ITS1, i.ITS2, i."5.8S"
            FROM base_pairs p
            LEFT JOIN its_wide i
              ON i.genome1 = least(p.genome1, p.genome2)
             AND i.genome2 = greatest(p.genome1, p.genome2)
            """
        ).fetchdf()
    finally:
        con.close()


In [3]:
def calculate_family_auprg(pairs):
    complete = pairs.dropna(subset=METRICS).copy()
    rows = []
    for (phylum, family), family_pairs in complete.groupby(
        ['phylum', 'family'], observed=True, sort=True
    ):
        labels = family_pairs['same_species'].to_numpy(dtype=np.int8)
        positives = int(labels.sum())
        negatives = int(labels.size - positives)
        if positives == 0 or negatives == 0:
            continue
        for metric in METRICS:
            rows.append({
                'phylum': phylum,
                'family': family,
                'metric': metric,
                'auPRG': float(average_precision_recall_gain(
                    labels, family_pairs[metric].to_numpy(dtype=float)
                )),
                'comparisons': labels.size,
            })
    results = pd.DataFrame(rows)
    if results.empty:
        raise RuntimeError('No eligible families with both comparison classes.')
    return results


def make_report_table(results):
    rows = []
    index = []
    for (phylum, family), group in results.groupby(
        ['phylum', 'family'], observed=True, sort=True
    ):
        scores = group.set_index('metric')['auPRG'].reindex(METRICS)
        comparisons = int(group['comparisons'].iloc[0])
        formatted_scores = [f'{value:.4f}' for value in scores]
        rows.extend([[str(comparisons)] * len(METRICS), formatted_scores])
        index.extend([
            (phylum, family, 'comparisons'),
            (phylum, family, 'auPRG'),
        ])
    return pd.DataFrame(
        rows,
        index=pd.MultiIndex.from_tuples(
            index, names=['phylum', 'family', 'statistic']
        ),
        columns=METRICS,
    )


In [4]:
pairs = load_filtered_pairs()
results = calculate_family_auprg(pairs)
report_table = make_report_table(results)
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
report_table.to_csv(RESULTS_PATH, sep='\t')
report_table


ANI     AAI    ITS1    ITS2  \
phylum     family             statistic                                     
Ascomycota Aspergillaceae     comparisons  106425  106425  106425  106425   
                              auPRG        0.6002  0.7133  0.3661  0.4527   
           Debaryomycetaceae  comparisons    2589    2589    2589    2589   
                              auPRG        0.9997  0.9997  1.0000  0.9798   
           Nectriaceae        comparisons    2576    2576    2576    2576   
                              auPRG        0.9879  0.9892  0.8794  0.9180   
           Pichiaceae         comparisons    1616    1616    1616    1616   
                              auPRG        0.9996  0.9995  0.9950  0.9660   
           Saccharomycetaceae comparisons  412596  412596  412596  412596   
                              auPRG        0.9508  0.9267  0.5222  0.1745   
Euglenozoa Trypanosomatidae   comparisons     379     379     379     379   
                              auPRG        0.3642  0.3916  0.5235  0.2735   
Oomycota   Peronosporaceae    comparisons     812     812     812     812   
                              auPRG        0.9991  1.0000  0.6211  0.9177   

                                             5.8S  
phylum     family             statistic            
Ascomycota Aspergillaceae     comparisons  106425  
                              auPRG        0.5609  
           Debaryomycetaceae  comparisons    2589  
                              auPRG        0.8733  
           Nectriaceae        comparisons    2576  
                              auPRG        0.5000  
           Pichiaceae         comparisons    1616  
                              auPRG        0.4833  
           Saccharomycetaceae comparisons  412596  
                              auPRG        0.3404  
Euglenozoa Trypanosomatidae   comparisons     379  
                              auPRG        0.4484  
Oomycota   Peronosporaceae    comparisons     812  
                              auPRG        0.8003